# mtrl on Colab: run experiments

The experiment is a training YAML; this notebook only runs it. To use it:

1. **Runtime → Change runtime type**: **L4** (recommended, bf16) or A100 40/80 GB.
   T4 only does fp16, which may overflow with Qwen3.5.
2. Edit the **first code cell** (`REF`, `CONFIGS`, `EVAL`).
3. **Runtime → Run all.**

What happens (`scripts/run_experiments.py`):

| Step | What |
|---|---|
| setup | `setup/colab_setup.sh`: packages, scorer and mtdata venvs, env config for this GPU (skips what is already there) |
| data | builds the splits for the directions the configs use; cached on Drive (`mtrl/data_cache`), so later sessions skip it |
| train | each config in order; runs and checkpoints are on Drive (`mtrl/runs`, `mtrl/ckpt`) |
| evaluate | greedy evaluation on the config's eval split (NTREX-128 `ntrex_dev` by default) |
| compare | one table over these runs |

**Disconnected?** Open the notebook again and Run all: finished steps are skipped and
training resumes from its last checkpoint. Set `schedule.max_hours` in the config below
the session length to stop cleanly with a checkpoint.

**Trying a variation without committing it:** write a YAML on Drive that inherits from
one in the repo, and put its path in `CONFIGS`:

```yaml
# /content/drive/MyDrive/mtrl/configs/try_lr.yaml
base: /content/mtrl/configs/train/ym/wide_all_ref.yaml
run_name: try_lr_5e-5
optim: {lr: 5.0e-5}
```

The merged config and the git commit end up in `mtrl/runs/<run_name>/config.resolved.yaml`.
Move configs worth keeping into `configs/train/` in the repo.

**Speed:** Colab's CPU makes small batches and token-by-token generation about 3x slower
than the local server; RL runs are better there (docs/training.md, Platforms).

In [ ]:
# ---------------- edit this cell only ----------------
REF = 'train/merge-ym'        # branch, tag or commit of the code (a commit pins the run)
CONFIGS = [                   # training YAMLs, run in order: repo paths or files on Drive
    'configs/train/quick_sft.yaml',
]
EVAL = True                   # evaluate each run after training
# ------------------------------------------------------

## Drive and code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
for d in ('runs', 'ckpt', 'data_cache'):
    os.makedirs(f'/content/drive/MyDrive/mtrl/{d}', exist_ok=True)

%cd /content
if not os.path.isdir('/content/mtrl/.git'):
    !git clone -q https://github.com/ETHz-DL-team29/mtrl.git /content/mtrl
!git -C /content/mtrl fetch -q && git -C /content/mtrl checkout -q {REF} && (git -C /content/mtrl pull -q --ff-only 2>/dev/null || true)
%cd /content/mtrl
!git log --oneline -1

## Setup (idempotent)

In [ ]:
!bash setup/colab_setup.sh
import sys
sys.path.insert(0, '/content/mtrl/src')   # this kernel started before mtrl was installed
os.environ['MTRL_ENV'] = open('.env').read().split('=', 1)[1].strip()
print('MTRL_ENV =', os.environ['MTRL_ENV'])

## Data, training, evaluation

In [ ]:
args = ' '.join(CONFIGS) + ('' if EVAL else ' --no-eval')
!python scripts/run_experiments.py {args} --data-cache /content/drive/MyDrive/mtrl/data_cache

## Training curves

In [ ]:
import json, pathlib
import matplotlib.pyplot as plt
from mtrl.common.train_config import load_train_config

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for cfg in CONFIGS:
    name = load_train_config(cfg).train.run_name
    run = pathlib.Path('/content/drive/MyDrive/mtrl/runs') / name
    tl = [json.loads(l) for l in (run / 'train_log.jsonl').read_text().splitlines()]
    key = 'reward' if 'reward' in tl[0] else 'loss'
    axes[0].plot([x['step'] for x in tl], [x[key] for x in tl], label=f'{name} ({key})')
    vp = run / 'val_log.jsonl'
    vl = [v for v in map(json.loads, vp.read_text().splitlines()) if 'val_chrf' in v] if vp.exists() else []
    axes[1].plot([v['step'] for v in vl], [v['val_chrf'] for v in vl], marker='o', label=name)
axes[0].set_title('training: reward (RL) or loss (SFT)'); axes[0].set_xlabel('step'); axes[0].legend()
axes[1].set_title('validation chrF++ (ntrex_val)'); axes[1].set_xlabel('step'); axes[1].legend()
plt.tight_layout(); plt.show()